# IndoBERT XAI Sentiment Analysis — Full Walkthrough

This notebook demonstrates:
1. Loading the fine-tuned IndoBERT model
2. Running sentiment prediction on diverse Indonesian texts
3. Comparing all 3 XAI methods side-by-side
4. Computing and displaying faithfulness metrics
5. **Baseline model comparison** (BiLSTM, mBERT, XLM-R vs IndoBERT) — Table 2
6. Case study: informal Indonesian where attention diverges from IG

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import config
print(f'Device: {config.DEVICE}')
print(f'Model: {config.MODEL_NAME}')

## 1. Load Fine-Tuned Model

In [ ]:
from src.model import load_best_model

model, tokenizer = load_best_model()
print('Model loaded successfully.')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 2. Sentiment Prediction on 5 Example Texts

In [ ]:
from src.model import predict_proba

# 5 examples: formal, informal, slang, emoticons, code-switching
example_texts = [
    # 1. Formal positive
    "Produk ini memiliki kualitas yang sangat baik dan harga yang terjangkau.",
    # 2. Informal slang negative
    "Ih nyebelin banget sih, pesanan gue salah mulu. Kapok deh beli disini!",
    # 3. With emoticons
    "Pelayanannya ramah banget :) Pasti balik lagi deh ke sini :D",
    # 4. Code-switching (mixed Indonesian-English)
    "The food was amazing! Tempatnya juga cozy dan Instagrammable banget.",
    # 5. Neutral / ambiguous
    "Biasa aja sih, tidak terlalu buruk tapi juga tidak istimewa.",
]

probs = predict_proba(model, tokenizer, example_texts)
pred_classes = probs.argmax(dim=-1).numpy()

df_preds = pd.DataFrame({
    'text': [t[:60] + ('...' if len(t) > 60 else '') for t in example_texts],
    'predicted': [config.LABEL_MAP[c] for c in pred_classes],
    'p_negative': probs[:, 0].numpy(),
    'p_neutral': probs[:, 1].numpy(),
    'p_positive': probs[:, 2].numpy(),
})
df_preds.style.format({'p_negative': '{:.3f}', 'p_neutral': '{:.3f}', 'p_positive': '{:.3f}'})

## 3. Compare All 3 XAI Methods Side-by-Side

In [ ]:
from src.xai.attention_viz import get_attention_weights, get_token_importance_from_attention
from src.xai.lime_explainer import IndoBERTLimeExplainer
from src.xai.ig_explainer import compute_ig_attributions
from src.visualize import plot_method_comparison

lime_exp = IndoBERTLimeExplainer(model, tokenizer, num_samples=300)
print('Explainers initialized.')

In [ ]:
# Analyze each example
for i, text in enumerate(example_texts):
    print(f'\n--- Example {i+1}: {text[:70]}...')
    
    # Attention
    attn_data = get_attention_weights(model, tokenizer, text)
    attn_imp = get_token_importance_from_attention(attn_data, strategy='rollout')
    
    raw_probs = torch.softmax(torch.tensor(attn_data['logits']), dim=-1).numpy()[0]
    pred_class = int(raw_probs.argmax())
    pred_label = config.LABEL_MAP[pred_class]
    print(f'  Prediction: {pred_label} (confidence: {raw_probs[pred_class]:.3f})')
    
    # LIME
    lime_explanation = lime_exp.explain_instance(text)
    lime_weights = lime_exp.get_lime_feature_weights(lime_explanation)
    
    # IG
    ig_data = compute_ig_attributions(model, tokenizer, text, target_class=pred_class)
    
    # Plot comparison
    fig = plot_method_comparison(
        attn_data['tokens'], attn_imp,
        lime_weights,
        ig_data['tokens'], ig_data['attributions_signed'],
        text=text,
        pred_label=pred_label,
        top_k=8,
    )
    plt.show()
    plt.close(fig)

## 4. Faithfulness Metric Table

In [ ]:
import os

metrics_path = config.RESULTS_DIR / 'xai_evaluation.csv'

if metrics_path.exists():
    df_metrics = pd.read_csv(metrics_path, index_col=0)
    print('Loaded pre-computed evaluation metrics.')
else:
    print('Metrics file not found. Running evaluation on 20 samples (this may take a while)...')
    from src.evaluate_xai import evaluate_all_methods
    from datasets import load_dataset
    dataset = load_dataset(config.DATASET_NAME, config.DATASET_SUBSET)
    test_texts = [str(ex['text']) for ex in dataset['test']]
    df_metrics, jaccard = evaluate_all_methods(model, tokenizer, test_texts, n_samples=20)
    df_metrics.to_csv(metrics_path)

# Display formatted
display_cols = ['aopc_mean', 'comprehensiveness_mean', 'sufficiency_mean']
df_metrics[display_cols].style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=0)

In [ ]:
# Radar chart visualization
from src.visualize import plot_metrics_radar

if 'df_metrics' in dir():
    fig = plot_metrics_radar(df_metrics)
    plt.show()
    plt.close(fig)

## 4b. Classification Performance Comparison (Table 2)

Load test results from IndoBERT and all baseline models (BiLSTM, mBERT, XLM-R)
to produce the Table 2 comparison for the paper.

In [ ]:
import json

# Map of model key -> (display name, results filename)
MODEL_FILES = {
    'BiLSTM':    config.RESULTS_DIR / 'test_results_bilstm.json',
    'mBERT':     config.RESULTS_DIR / 'test_results_mbert.json',
    'XLM-R base': config.RESULTS_DIR / 'test_results_xlmr.json',
    'IndoBERT':  config.RESULTS_DIR / 'test_results.json',
}

rows = []
missing = []
for name, path in MODEL_FILES.items():
    if path.exists():
        with open(path) as f:
            res = json.load(f)
        rows.append({
            'Model': name,
            'Accuracy': res.get('test_accuracy', res.get('eval_accuracy')),
            'Macro-F1': res.get('test_macro_f1', res.get('eval_macro_f1')),
            'F1 (Pos)': res.get('test_f1_positive', res.get('eval_f1_positive')),
            'F1 (Neu)': res.get('test_f1_neutral', res.get('eval_f1_neutral')),
            'F1 (Neg)': res.get('test_f1_negative', res.get('eval_f1_negative')),
        })
    else:
        missing.append(name)

if missing:
    print(f'⚠ Results not found for: {", ".join(missing)}')
    print(f'  Run training first:')
    if 'IndoBERT' in missing:
        print(f'    python src/train.py')
    baseline_missing = [m for m in missing if m != 'IndoBERT']
    if baseline_missing:
        key_map = {'BiLSTM': 'bilstm', 'mBERT': 'mbert', 'XLM-R base': 'xlmr'}
        for m in baseline_missing:
            print(f'    python src/train_baselines.py --model {key_map[m]}')

if rows:
    df_table2 = pd.DataFrame(rows).set_index('Model')
    # Reorder to match paper table
    order = ['BiLSTM', 'mBERT', 'XLM-R base', 'IndoBERT']
    df_table2 = df_table2.reindex([m for m in order if m in df_table2.index])
    
    print('\nTable 2: Classification Performance on SmSA Test Set\n')
    display(
        df_table2.style
        .format('{:.4f}')
        .highlight_max(axis=0, props='font-weight: bold; color: #2e7d32')
    )
else:
    print('No results files found. Train models first.')

In [ ]:
# Bar chart: Accuracy and Macro-F1 across models
if 'df_table2' in dir() and not df_table2.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    colors = ['#90a4ae', '#7986cb', '#4fc3f7', '#66bb6a']
    n_models = len(df_table2)
    
    # Accuracy
    ax = axes[0]
    bars = ax.bar(range(n_models), df_table2['Accuracy'], color=colors[:n_models])
    ax.set_xticks(range(n_models))
    ax.set_xticklabels(df_table2.index, rotation=15)
    ax.set_ylabel('Accuracy')
    ax.set_title('Test Accuracy')
    ax.set_ylim(bottom=max(0, df_table2['Accuracy'].min() - 0.1))
    for bar, val in zip(bars, df_table2['Accuracy']):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Macro-F1
    ax = axes[1]
    bars = ax.bar(range(n_models), df_table2['Macro-F1'], color=colors[:n_models])
    ax.set_xticks(range(n_models))
    ax.set_xticklabels(df_table2.index, rotation=15)
    ax.set_ylabel('Macro-F1')
    ax.set_title('Test Macro-F1')
    ax.set_ylim(bottom=max(0, df_table2['Macro-F1'].min() - 0.1))
    for bar, val in zip(bars, df_table2['Macro-F1']):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    fig.suptitle('Table 2: Classification Performance Comparison', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # Per-class F1 grouped bar chart
    f1_cols = ['F1 (Neg)', 'F1 (Neu)', 'F1 (Pos)']
    if all(c in df_table2.columns for c in f1_cols):
        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(n_models)
        width = 0.25
        class_colors = ['#ef5350', '#ffa726', '#66bb6a']
        
        for j, (col, color) in enumerate(zip(f1_cols, class_colors)):
            bars = ax.bar(x + j * width, df_table2[col], width, label=col, color=color)
            for bar, val in zip(bars, df_table2[col]):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=8)
        
        ax.set_xticks(x + width)
        ax.set_xticklabels(df_table2.index)
        ax.set_ylabel('F1 Score')
        ax.set_title('Per-Class F1 Scores by Model')
        ax.set_ylim(bottom=max(0, df_table2[f1_cols].min().min() - 0.1))
        ax.legend()
        plt.tight_layout()
        plt.show()
        plt.close(fig)
else:
    print('No results to plot.')

## 5. Case Study: Informal Indonesian — Attention vs. IG Divergence

Indonesian informal slang (e.g., 'gue', 'banget', 'dong', shortened forms) can cause
attention to focus on surface features (repeated chars, colloquial terms) while
IG captures deeper semantic attribution patterns.

In [ ]:
## Summary

Key findings:
- **Baseline comparison**: IndoBERT outperforms BiLSTM, mBERT, and XLM-R on SmSA sentiment classification (Table 2), confirming the advantage of language-specific pre-training
- **Attention rollout** tends to spread importance across syntactic boundary tokens
- **LIME** identifies discriminative words at the word/phrase level, handling slang well
- **IG** provides gradient-based attribution that can distinguish affixed forms
- Informal expressions and emoticons cause divergence between methods — a productive
  area for future research into Indonesian-specific XAI calibration

In [ ]:
# Visualize divergence
fig = plot_method_comparison(
    attn_data['tokens'], attn_imp,
    lime_weights_cs,
    ig_data_cs['tokens'], ig_data_cs['attributions_signed'],
    text=case_study_text,
    pred_label=config.LABEL_MAP[pred_class],
    top_k=10,
)
plt.suptitle('Case Study: Informal Indonesian — Method Divergence', fontsize=12, y=1.02)
plt.show()
plt.close(fig)

In [ ]:
# Compute Jaccard overlap between methods for this case study
from src.evaluate_xai import token_overlap_jaccard, _get_word_positions

def top_k_tokens_from_imp(tokens, imp, k=10):
    word_pos = _get_word_positions(tokens)
    ranked = sorted(word_pos, key=lambda i: abs(imp[i]), reverse=True)
    return [tokens[i] for i in ranked[:k]]

lime_imp_cs = np.array([
    abs(lime_weights_cs.get(tok.replace('##', ''), 0.0))
    for tok in attn_data['tokens']
])

attn_top10 = top_k_tokens_from_imp(attn_data['tokens'], attn_imp)
lime_top10 = top_k_tokens_from_imp(attn_data['tokens'], lime_imp_cs)
ig_top10 = top_k_tokens_from_imp(ig_data_cs['tokens'], np.abs(ig_data_cs['attributions_signed']))

print('Token Overlap (Jaccard) for case study:')
print(f'  Attention vs LIME: {token_overlap_jaccard(attn_top10, lime_top10):.3f}')
print(f'  Attention vs IG:   {token_overlap_jaccard(attn_top10, ig_top10):.3f}')
print(f'  LIME vs IG:        {token_overlap_jaccard(lime_top10, ig_top10):.3f}')
print('\nNote: Lower overlap indicates method divergence — common for informal slang.')

## Summary

Key findings:
- **Attention rollout** tends to spread importance across syntactic boundary tokens
- **LIME** identifies discriminative words at the word/phrase level, handling slang well
- **IG** provides gradient-based attribution that can distinguish affixed forms
- Informal expressions and emoticons cause divergence between methods — a productive
  area for future research into Indonesian-specific XAI calibration